# P1-E07. Cohort CAUEEG Reference Preprocessing and Automated Epoch-QA Lock

**Paper 1, Experiment 07. full-cohort successor to the passed P1-E06. automated preprocessing-path lock**

## Goal

Apply the frozen **reference** preprocessing path to all 979 P1-E05. channel-harmonised recordings and perform fully automatic interval/epoch signal-quality control. No EEG-specialist form or manual rating is required.

This notebook:

1. authenticates the completed P1-E06.1 lock and its P1-E05.1 source manifests;
2. reads the exact locked 19-channel indices and 8,557 event-defined intervals;
3. detrends and filters each interval from 1–40 Hz without applying a second average reference;
4. constructs non-overlapping 10-second epochs after a two-second interval-edge trim;
5. applies the frozen automatic-QA rules without diagnosis labels;
6. retains up to 12 temporally distributed accepted epochs per recording, requiring at least six;
7. saves one compressed epoch array per eligible recording plus complete interval, epoch and recording audit tables; and
8. joins diagnosis only after preprocessing eligibility is frozen.

Run all cells in order. Continue only if the final status is:

> **P1-E07. PASSED — FULL-COHORT PREPROCESSED EPOCH MANIFEST LOCKED**

## Context & Methods

### Frozen automatic-QA rules

- Input path: P1-E06.1 selected `reference`.
- Input signals: the released CAUEEG `-AVG` channels in the P1-E05.1 canonical order; no second common-average reference.
- Filter: fourth-order Butterworth, zero-phase, 1–40 Hz.
- Target sampling rate: 200 Hz.
- Interval edge trim: 2 seconds at each end.
- Epoch length: 10 seconds, 0% overlap.
- Hard epoch limits: peak-to-peak <600 µV, RMS <120 µV and extreme-amplitude fraction <2% at |150| µV.
- Multimetric robust outlier score: <5, calculated within each recording from peak-to-peak amplitude, RMS, maximum step and extreme-amplitude fraction.
- Non-finite epochs and epochs with a numerically flat channel are rejected.
- Up to 12 accepted epochs are selected deterministically across the recording timeline; at least six are required.
- Processing and QA are diagnosis-independent. Labels and splits are joined only after the recording-level eligibility decision is frozen.
- The full-cohort lock requires at least 90% of input recordings to remain eligible and no more than 5% processing failures.
- Manual specialist review is not required. Independent expert validation remains a future recommendation only.

## Setup

### 1. Paths, parameters and helpers

In [ ]:
from pathlib import Path
from fractions import Fraction
import hashlib, json, platform, sys, traceback, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from IPython.display import display

try:
    import mne
except Exception as exc:
    raise ImportError("P1-E07.1 requires MNE. Install with: conda install -c conda-forge mne -y") from exc

PAPER_ROOT=Path(r'D:\EEG_AD_Research\paper1_experiments\paper1_outputs')
E05_ROOT=PAPER_ROOT/'p1_e05_1_channel_montage_lock'
E06_ROOT=PAPER_ROOT/'p1_e06_1_automated_qa_preprocessing_pilot'
OUTPUT_ROOT=PAPER_ROOT/'p1_e07_1_full_cohort_reference_preprocessing_automated_qa'
TABLES=OUTPUT_ROOT/'tables'; FIGURES=OUTPUT_ROOT/'figures'; ARRAYS=OUTPUT_ROOT/'epoch_arrays'
for path in (OUTPUT_ROOT,TABLES,FIGURES,ARRAYS): path.mkdir(parents=True,exist_ok=True)

E06_RUN_RECORD=E06_ROOT/'run_record.json'
E06_DECISION=E06_ROOT/'p1_e06_1_automated_qa_preprocessing_path_lock.json'
E06_SPEC=E06_ROOT/'tables'/'10_LOCKED_AUTOMATED_PREPROCESSING_SPECIFICATION.csv'
E06_HANDOFF=E06_ROOT/'tables'/'11_P1_E07_FULL_COHORT_AUTOMATED_QA_HANDOFF.csv'
E05_RECORDINGS=E05_ROOT/'tables'/'13_LOCKED_CHANNEL_HARMONISED_RECORDING_MANIFEST.csv'
E05_INTERVALS=E05_ROOT/'tables'/'14_LOCKED_CHANNEL_HARMONISED_INTERVAL_MANIFEST.csv'
E05_CHANNEL_MAP=E05_ROOT/'tables'/'12_LOCKED_CANONICAL_19_EEG_CHANNEL_SELECTION_MAP.csv'

CANONICAL=['Fp1','Fp2','F7','F3','Fz','F4','F8','T3','C3','Cz','C4','T4','T5','P3','Pz','P4','T6','O1','O2']
FS_TARGET=200.0; BAND=(1.0,40.0); FILTER_ORDER=4
EDGE_TRIM_SECONDS=2.0; EPOCH_SECONDS=10.0; TARGET_EPOCHS=12; MIN_EPOCHS=6
PTP_LIMIT_UV=600.0; RMS_LIMIT_UV=120.0; EXTREME_UV=150.0
MAX_EXTREME_FRACTION=0.02; ROBUST_SCORE_LIMIT=5.0; FLAT_SD_UV=1e-6
MIN_COHORT_RETENTION=0.90; MAX_PROCESSING_FAILURE_FRACTION=0.05
CLASS_ORDER=['Normal','MCI','AD']; SPLIT_ORDER=['train','validation','test']

plt.rcParams.update({'figure.dpi':120,'savefig.dpi':600,'font.size':9,
                     'axes.titlesize':10,'axes.labelsize':9,'pdf.fonttype':42,
                     'ps.fonttype':42,'savefig.bbox':'tight'})
sns.set_theme(style='whitegrid',context='paper'); mne.set_log_level('ERROR')

def sha256_file(path,block_size=4*1024*1024):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(block_size),b''): digest.update(block)
    return digest.hexdigest()
def save_table(frame,name):
    frame.to_csv(TABLES/f'{name}.csv',index=False)
    try: frame.to_excel(TABLES/f'{name}.xlsx',index=False)
    except Exception as exc: warnings.warn(f'Excel export skipped ({exc}); CSV was saved.')
def savefig(fig,name,width=8,height=4):
    fig.set_size_inches(width,height); fig.tight_layout()
    fig.savefig(FIGURES/f'{name}.png',dpi=600,bbox_inches='tight',facecolor='white')
    fig.savefig(FIGURES/f'{name}.pdf',bbox_inches='tight',facecolor='white')
def robust_z(values):
    values=np.asarray(values,float); median=np.nanmedian(values)
    mad=np.nanmedian(np.abs(values-median))
    if not np.isfinite(mad) or mad<1e-12: return np.zeros_like(values)
    return (values-median)/(1.4826*mad)

print('Experiment: P1-E07.1')
print('Input P1-E06.1:',E06_ROOT)
print('Output:',OUTPUT_ROOT)

## Data

### 2. Authenticate P1-E06.1 and all inherited locks

In [ ]:
required=[E06_RUN_RECORD,E06_DECISION,E06_SPEC,E06_HANDOFF,
          E05_RECORDINGS,E05_INTERVALS,E05_CHANNEL_MAP]
missing=[str(path) for path in required if not path.is_file()]
if missing: raise FileNotFoundError('Required locked inputs are missing:\n'+'\n'.join(missing))

e06_run=json.loads(E06_RUN_RECORD.read_text(encoding='utf-8'))
decision=json.loads(E06_DECISION.read_text(encoding='utf-8'))
spec=pd.read_csv(E06_SPEC)
handoff=pd.read_csv(E06_HANDOFF,dtype={'recording_id':str,'serial':str})
recordings=pd.read_csv(E05_RECORDINGS,dtype={'recording_id':str,'serial':str})
intervals=pd.read_csv(E05_INTERVALS,dtype={'recording_id':str,'serial':str})
channel_map=pd.read_csv(E05_CHANNEL_MAP,dtype={'recording_id':str,'serial':str})
for frame in (handoff,recordings,intervals,channel_map):
    for column in ('recording_id','serial'):
        if column in frame: frame[column]=frame[column].astype(str).str.strip().str.zfill(5)

expected_hashes=e06_run.get('input_hashes',{})
observed_hashes={'recordings':sha256_file(E05_RECORDINGS),'intervals':sha256_file(E05_INTERVALS),
                 'channel_map':sha256_file(E05_CHANNEL_MAP)}
canonical_orders=(channel_map.sort_values(['recording_id','canonical_index'])
                  .groupby('recording_id').canonical_name.apply(list))

input_checks=pd.DataFrame([
 {'check':'P1-E06.1 status passed','pass':str(e06_run.get('status','')).startswith('P1-E06.1 PASSED'),'detail':e06_run.get('status','')},
 {'check':'reference path selected','pass':decision.get('selected_primary_path')=='reference' and e06_run.get('selected_primary_path')=='reference','detail':decision.get('selected_primary_path')},
 {'check':'manual specialist review not required','pass':decision.get('manual_review_required') is False and e06_run.get('manual_specialist_review_required') is False,'detail':'automatic QA'},
 {'check':'P1-E06.1 decision authenticated','pass':sha256_file(E06_DECISION)==e06_run.get('decision_sha256'),'detail':sha256_file(E06_DECISION)},
 {'check':'P1-E06.1 specification authenticated','pass':sha256_file(E06_SPEC)==e06_run.get('locked_specification_sha256'),'detail':sha256_file(E06_SPEC)},
 {'check':'P1-E07 handoff authenticated','pass':sha256_file(E06_HANDOFF)==e06_run.get('p1_e07_handoff_sha256'),'detail':sha256_file(E06_HANDOFF)},
 {'check':'inherited P1-E05.1 hashes authenticated','pass':all(expected_hashes.get(k)==v for k,v in observed_hashes.items()),'detail':observed_hashes},
 {'check':'979 recordings and 8,557 intervals','pass':len(recordings)==979 and len(intervals)==8557,'detail':{'recordings':len(recordings),'intervals':len(intervals)}},
 {'check':'handoff recording set exact','pass':set(handoff.recording_id)==set(recordings.recording_id) and len(handoff)==979,'detail':len(handoff)},
 {'check':'19 locked map rows per recording','pass':channel_map.groupby('recording_id').size().eq(19).all() and channel_map.recording_id.nunique()==979,'detail':channel_map.groupby('recording_id').size().value_counts().to_dict()},
 {'check':'canonical order exact','pass':canonical_orders.map(lambda x:x==CANONICAL).all(),'detail':'19-channel order'},
 {'check':'analysis remains recording-level','pass':recordings.analysis_unit.astype(str).str.lower().eq('recording').all(),'detail':recordings.analysis_unit.value_counts().to_dict()},
 {'check':'participant identity remains unverified','pass':~recordings.participant_identity_verified.astype(str).str.lower().isin(['true','1']).any(),'detail':recordings.participant_identity_verified.value_counts(dropna=False).to_dict()},
])
save_table(input_checks,'01_input_authentication_checks'); display(input_checks)
if not input_checks['pass'].all(): raise AssertionError('STOP — P1-E06.1 INPUT AUTHENTICATION FAILED')
print('P1-E06.1 AND INHERITED P1-E05.1 LOCKS AUTHENTICATED')

## Results

### 3. Define locked preprocessing and automatic epoch-QA functions

In [ ]:
def read_locked_channels(edf_path,recording_id):
    raw=mne.io.read_raw_edf(str(edf_path),preload=False,verbose='ERROR')
    rows=(channel_map.loc[channel_map.recording_id.eq(recording_id)]
          .sort_values('canonical_index'))
    if len(rows)!=19 or rows.canonical_name.tolist()!=CANONICAL:
        raw.close(); raise ValueError('locked_channel_map_invalid')
    picks=rows.original_index.astype(int).tolist()
    if max(picks)>=len(raw.ch_names):
        raw.close(); raise ValueError('locked_channel_index_outside_header')
    if [raw.ch_names[index] for index in picks]!=rows.original_name.tolist():
        raw.close(); raise ValueError('edf_header_changed_after_channel_lock')
    return raw,picks

def preprocess_interval(data_uv,source_fs):
    data_uv=signal.detrend(np.asarray(data_uv,float),axis=1,type='linear')
    sos=signal.butter(FILTER_ORDER,BAND,btype='bandpass',fs=source_fs,output='sos')
    data_uv=signal.sosfiltfilt(sos,data_uv,axis=1)
    if not np.isclose(source_fs,FS_TARGET):
        ratio=Fraction(FS_TARGET/source_fs).limit_denominator(1000)
        data_uv=signal.resample_poly(data_uv,ratio.numerator,ratio.denominator,axis=1)
    return data_uv

def make_interval_epochs(processed,interval_number,start_seconds):
    trim=int(round(EDGE_TRIM_SECONDS*FS_TARGET)); length=int(round(EPOCH_SECONDS*FS_TARGET))
    if processed.shape[1]<=2*trim: return [],[]
    usable=processed[:,trim:processed.shape[1]-trim]
    epochs=[]; metadata=[]
    for offset in range(0,usable.shape[1]-length+1,length):
        epochs.append(usable[:,offset:offset+length])
        metadata.append({'interval_number':int(interval_number),
            'epoch_start_seconds':float(start_seconds+EDGE_TRIM_SECONDS+offset/FS_TARGET),
            'epoch_stop_seconds':float(start_seconds+EDGE_TRIM_SECONDS+(offset+length)/FS_TARGET)})
    return epochs,metadata

def epoch_metrics(epochs):
    finite=np.isfinite(epochs).all(axis=(1,2))
    channel_sd=np.std(epochs,axis=2)
    flat_channels=(channel_sd<=FLAT_SD_UV).sum(axis=1)
    ptp=np.ptp(epochs,axis=2).max(axis=1)
    rms=np.sqrt(np.mean(epochs**2,axis=(1,2)))
    max_step=np.max(np.abs(np.diff(epochs,axis=2)),axis=(1,2))
    extreme_fraction=np.mean(np.abs(epochs)>=EXTREME_UV,axis=(1,2))
    robust=np.maximum.reduce([
        np.abs(robust_z(np.log(ptp+1e-9))),np.abs(robust_z(np.log(rms+1e-9))),
        np.abs(robust_z(np.log(max_step+1e-9))),np.abs(robust_z(extreme_fraction+1e-9))])
    accepted=(finite & (flat_channels==0) & (ptp<PTP_LIMIT_UV) &
              (rms<RMS_LIMIT_UV) & (extreme_fraction<MAX_EXTREME_FRACTION) &
              (robust<ROBUST_SCORE_LIMIT))
    return pd.DataFrame({'finite':finite,'flat_channels':flat_channels,'ptp_uv':ptp,
        'rms_uv':rms,'max_step_uv':max_step,'extreme_fraction':extreme_fraction,
        'robust_multimetric_score':robust,'accepted':accepted})

def evenly_distributed_indices(indices,target):
    indices=np.asarray(indices,dtype=int)
    if len(indices)<=target: return indices
    positions=np.linspace(0,len(indices)-1,target).round().astype(int)
    return indices[positions]

print('Frozen path: reference | no second CAR | 1–40 Hz | 10-s non-overlapping epochs')

### 4. Process all recordings and save diagnosis-independent QA outputs

This is the computationally expensive cell. Progress is printed every 25 recordings. Existing per-recording arrays are overwritten deterministically when the cell is rerun.

In [ ]:
blind_recordings=recordings[['recording_id','serial','edf_path']].copy()
interval_audit_rows=[]; epoch_audit_rows=[]; recording_audit_rows=[]; failures=[]

for number,row in enumerate(blind_recordings.itertuples(index=False),1):
    raw=None
    try:
        raw,picks=read_locked_channels(row.edf_path,row.recording_id)
        source_fs=float(raw.info['sfreq'])
        record_intervals=(intervals.loc[intervals.recording_id.eq(row.recording_id)]
                          .sort_values(['start_seconds','interval_number']))
        epoch_list=[]; meta_list=[]
        for interval in record_intervals.itertuples(index=False):
            start=max(0,int(round(float(interval.start_seconds)*source_fs)))
            stop=min(raw.n_times,int(round(float(interval.stop_seconds)*source_fs)))
            try:
                data=raw.get_data(picks=picks,start=start,stop=stop)*1e6
                processed=preprocess_interval(data,source_fs)
                new_epochs,new_meta=make_interval_epochs(
                    processed,interval.interval_number,float(interval.start_seconds))
                epoch_list.extend(new_epochs); meta_list.extend(new_meta)
                interval_audit_rows.append({'recording_id':row.recording_id,'serial':row.serial,
                    'interval_number':int(interval.interval_number),'source_samples':int(stop-start),
                    'candidate_epochs':len(new_epochs),'processing_success':True,'error':''})
            except Exception as interval_exc:
                interval_audit_rows.append({'recording_id':row.recording_id,'serial':row.serial,
                    'interval_number':int(interval.interval_number),'source_samples':int(max(stop-start,0)),
                    'candidate_epochs':0,'processing_success':False,
                    'error':f'{type(interval_exc).__name__}: {interval_exc}'})

        if not epoch_list: raise ValueError('no_candidate_epochs_after_locked_preprocessing')
        epochs=np.asarray(epoch_list,dtype=float)
        metrics=epoch_metrics(epochs)
        meta=pd.DataFrame(meta_list)
        meta.insert(0,'candidate_epoch_number',np.arange(1,len(meta)+1))
        epoch_table=pd.concat([meta,metrics],axis=1)
        epoch_table.insert(0,'serial',row.serial); epoch_table.insert(0,'recording_id',row.recording_id)
        accepted_positions=np.flatnonzero(metrics.accepted.to_numpy())
        selected_positions=evenly_distributed_indices(accepted_positions,TARGET_EPOCHS)
        epoch_table['selected_for_lock']=False
        if len(selected_positions): epoch_table.loc[selected_positions,'selected_for_lock']=True
        epoch_audit_rows.extend(epoch_table.to_dict('records'))

        eligible=len(selected_positions)>=MIN_EPOCHS
        array_path=''
        array_sha256=''
        if eligible:
            selected=epochs[selected_positions].astype('float32')
            array_file=ARRAYS/f'{row.recording_id}_reference_epochs.npz'
            np.savez_compressed(array_file,epochs_uv=selected,fs_Hz=np.float32(FS_TARGET),
                channels=np.asarray(CANONICAL,dtype='U8'),
                candidate_epoch_number=epoch_table.loc[selected_positions,'candidate_epoch_number'].to_numpy(dtype='int32'),
                interval_number=epoch_table.loc[selected_positions,'interval_number'].to_numpy(dtype='int32'),
                epoch_start_seconds=epoch_table.loc[selected_positions,'epoch_start_seconds'].to_numpy(dtype='float64'))
            array_path=str(array_file); array_sha256=sha256_file(array_file)
        reason='eligible' if eligible else 'fewer_than_6_accepted_epochs'
        recording_audit_rows.append({'recording_id':row.recording_id,'serial':row.serial,
            'source_sfreq_Hz':source_fs,'locked_intervals':len(record_intervals),
            'candidate_epochs':len(epochs),'accepted_epochs':int(metrics.accepted.sum()),
            'selected_epochs':len(selected_positions),'accepted_fraction':float(metrics.accepted.mean()),
            'median_ptp_uv':float(metrics.ptp_uv.median()),'median_rms_uv':float(metrics.rms_uv.median()),
            'median_extreme_fraction':float(metrics.extreme_fraction.median()),
            'processing_success':True,'preprocessing_eligible':eligible,'reason':reason,
            'array_path':array_path,'array_sha256':array_sha256})
    except Exception as exc:
        failures.append({'recording_id':row.recording_id,'serial':row.serial,
            'error_type':type(exc).__name__,'error':str(exc),'traceback':traceback.format_exc(limit=3)})
        recording_audit_rows.append({'recording_id':row.recording_id,'serial':row.serial,
            'source_sfreq_Hz':np.nan,'locked_intervals':0,'candidate_epochs':0,'accepted_epochs':0,
            'selected_epochs':0,'accepted_fraction':0.0,'median_ptp_uv':np.nan,'median_rms_uv':np.nan,
            'median_extreme_fraction':np.nan,'processing_success':False,
            'preprocessing_eligible':False,'reason':'processing_failure','array_path':'','array_sha256':''})
    finally:
        if raw is not None: raw.close()
    if number%25==0 or number==len(blind_recordings):
        print(f'[{number:03d}/{len(blind_recordings)}] recordings processed')

interval_audit=pd.DataFrame(interval_audit_rows)
epoch_audit=pd.DataFrame(epoch_audit_rows)
recording_audit=pd.DataFrame(recording_audit_rows)
failure_audit=pd.DataFrame(failures,columns=['recording_id','serial','error_type','error','traceback'])
save_table(interval_audit,'02_interval_preprocessing_audit')
save_table(epoch_audit,'03_candidate_epoch_automatic_qa_audit')
save_table(recording_audit,'04_diagnosis_independent_recording_preprocessing_eligibility')
save_table(failure_audit,'05_processing_failures')
display(recording_audit.reason.value_counts().rename_axis('reason').reset_index(name='recordings'))
display(recording_audit[['candidate_epochs','accepted_epochs','selected_epochs','accepted_fraction']].describe())

### 5. Freeze eligibility, then join labels and official splits

In [ ]:
audit_columns=['recording_id','serial','official_split','experimental_label','experimental_label_code',
               'age','analysis_unit','identity_group','participant_identity_verified','split_source',
               'edf_path','event_path','duration_seconds','sfreq_Hz','n_samples','edf_sha256',
               'event_sha256','manifest_row_sha256','retained_intervals','retained_seconds',
               'interval_manifest_version','channel_rule_version','canonical_layout']
audit_columns=[column for column in audit_columns if column in recordings.columns]
with_labels=recording_audit.merge(recordings[audit_columns],on=['recording_id','serial'],
                                  how='left',validate='one_to_one')
eligible_manifest=with_labels.loc[with_labels.preprocessing_eligible].copy()
selected_epochs=epoch_audit.loc[epoch_audit.selected_for_lock].copy()
selected_epoch_manifest=selected_epochs.merge(
    eligible_manifest[['recording_id','official_split','experimental_label','array_path','array_sha256']],
    on='recording_id',how='inner',validate='many_to_one')

summary=(with_labels.groupby(['official_split','experimental_label']).agg(
    input_recordings=('recording_id','size'),eligible_recordings=('preprocessing_eligible','sum'),
    median_candidate_epochs=('candidate_epochs','median'),median_accepted_epochs=('accepted_epochs','median'),
    median_acceptance=('accepted_fraction','median')).reset_index())
summary['retention_fraction']=summary.eligible_recordings/summary.input_recordings
reason_summary=(with_labels.groupby(['official_split','experimental_label','reason']).size()
                .rename('recordings').reset_index())

eligible_manifest['preprocessing_rule_version']='P1-E07.1-reference-autoQA-v1'
eligible_manifest['source_p1_e06_decision_sha256']=sha256_file(E06_DECISION)
selected_epoch_manifest['preprocessing_rule_version']='P1-E07.1-reference-autoQA-v1'

save_table(with_labels,'06_recording_preprocessing_audit_with_labels')
save_table(summary,'07_preprocessing_eligibility_by_split_and_class')
save_table(reason_summary,'08_exclusion_reasons_by_split_and_class')
save_table(eligible_manifest,'09_LOCKED_PREPROCESSING_ELIGIBLE_RECORDING_MANIFEST')
save_table(selected_epoch_manifest,'10_LOCKED_AUTOMATIC_QA_EPOCH_MANIFEST')
display(summary); display(reason_summary)

### 6. Publication-ready automatic-QA figures

In [ ]:
fig,axes=plt.subplots(1,2)
reason_counts=recording_audit.reason.value_counts().rename_axis('reason').reset_index(name='recordings')
sns.barplot(data=reason_counts,x='reason',y='recordings',hue='reason',legend=False,ax=axes[0],palette='Set2')
axes[0].set(title='Automatic preprocessing eligibility',xlabel='',ylabel='Recordings')
axes[0].tick_params(axis='x',rotation=25)
sns.histplot(data=recording_audit,x='accepted_fraction',bins=25,color='#0072B2',ax=axes[1])
axes[1].set(title='Automatic epoch acceptance',xlabel='Accepted candidate-epoch fraction',ylabel='Recordings')
savefig(fig,'Figure_01_full_cohort_automatic_qa',9,3.8); plt.show()

class_summary=(with_labels.groupby('experimental_label').agg(
    input_recordings=('recording_id','size'),eligible_recordings=('preprocessing_eligible','sum'))
    .reindex(CLASS_ORDER).reset_index())
fig,ax=plt.subplots(); x=np.arange(len(CLASS_ORDER))
ax.bar(x-.18,class_summary.input_recordings,.36,label='P1-E05.1 input',color='#BDBDBD')
ax.bar(x+.18,class_summary.eligible_recordings,.36,label='P1-E07.1 eligible',color='#009E73')
ax.set(xticks=x,xticklabels=CLASS_ORDER,ylabel='Recordings',title='Full-cohort preprocessing retention')
ax.legend(frameon=False)
for container in ax.containers: ax.bar_label(container,padding=2,fontsize=8)
savefig(fig,'Figure_02_preprocessing_retention_by_class',6,3.8); plt.show()

plot_data=epoch_audit.sample(min(len(epoch_audit),20000),random_state=20260828) if len(epoch_audit)>20000 else epoch_audit
fig,axes=plt.subplots(1,3)
for ax,metric,title in zip(axes,['ptp_uv','rms_uv','robust_multimetric_score'],
                           ['Peak-to-peak amplitude','RMS amplitude','Robust QA score']):
    sns.boxplot(data=plot_data,x='accepted',y=metric,hue='accepted',legend=False,showfliers=False,ax=ax)
    ax.set(title=title,xlabel='Epoch accepted',ylabel=metric)
savefig(fig,'Figure_03_epoch_quality_distributions',10,3.8); plt.show()

## Checks

### 7. Final integrity gates and reproducibility record

In [ ]:
input_count=len(recordings); eligible_count=len(eligible_manifest)
retention=eligible_count/input_count
failure_fraction=float((~recording_audit.processing_success).mean())
final_counts=(eligible_manifest.groupby(['official_split','experimental_label']).size().unstack(fill_value=0)
              .reindex(index=SPLIT_ORDER,columns=CLASS_ORDER,fill_value=0).astype(int))
selected_counts=selected_epoch_manifest.groupby('recording_id').size()
array_exists=eligible_manifest.array_path.map(lambda path:Path(path).is_file()).all()
array_hashes_match=all(sha256_file(row.array_path)==row.array_sha256
                       for row in eligible_manifest[['array_path','array_sha256']].itertuples(index=False))

checks=pd.DataFrame([
 {'check':'P1-E06.1 lock authenticated','pass':input_checks['pass'].all(),'detail':'all inherited gates passed'},
 {'check':'one recording audit row per input','pass':len(recording_audit)==979 and recording_audit.recording_id.nunique()==979,'detail':len(recording_audit)},
 {'check':'processing-failure fraction acceptable','pass':failure_fraction<=MAX_PROCESSING_FAILURE_FRACTION,'detail':failure_fraction},
 {'check':'cohort-retention threshold passed','pass':retention>=MIN_COHORT_RETENTION,'detail':retention},
 {'check':'selected epoch counts reconcile','pass':set(selected_counts.index)==set(eligible_manifest.recording_id) and selected_counts.between(MIN_EPOCHS,TARGET_EPOCHS).all(),'detail':selected_counts.value_counts().to_dict()},
 {'check':'only accepted epochs selected','pass':selected_epochs.accepted.all(),'detail':int((~selected_epochs.accepted).sum())},
 {'check':'all selected epochs are 10 seconds','pass':np.allclose(selected_epochs.epoch_stop_seconds-selected_epochs.epoch_start_seconds,EPOCH_SECONDS),'detail':EPOCH_SECONDS},
 {'check':'every eligible array exists','pass':array_exists,'detail':eligible_count},
 {'check':'every eligible array SHA-256 matches','pass':array_hashes_match,'detail':eligible_count},
 {'check':'all endpoint classes remain in every split','pass':final_counts.gt(0).all().all(),'detail':final_counts.to_dict()},
 {'check':'official splits preserved','pass':set(eligible_manifest.official_split).issubset(set(SPLIT_ORDER)),'detail':eligible_manifest.official_split.value_counts().to_dict()},
 {'check':'analysis remains recording-level','pass':eligible_manifest.analysis_unit.astype(str).str.lower().eq('recording').all(),'detail':eligible_manifest.analysis_unit.value_counts().to_dict()},
 {'check':'participant identity remains unverified','pass':~eligible_manifest.participant_identity_verified.astype(str).str.lower().isin(['true','1']).any(),'detail':eligible_manifest.participant_identity_verified.value_counts(dropna=False).to_dict()},
 {'check':'preprocessing and QA remained diagnosis-independent','pass':'experimental_label' not in blind_recordings.columns and 'experimental_label' not in recording_audit.columns and 'experimental_label' not in epoch_audit.columns,'detail':'labels joined after eligibility freeze'},
 {'check':'manual specialist review not required','pass':True,'detail':'future validation recommendation only'},
])
save_table(checks,'11_final_integrity_checks'); display(checks)

status=('P1-E07.1 PASSED — FULL-COHORT PREPROCESSED EPOCH MANIFEST LOCKED'
        if checks['pass'].all() else 'STOP — P1-E07.1 CHECKS FAILED; DO NOT BEGIN REPRESENTATION EXTRACTION')

recording_csv=TABLES/'09_LOCKED_PREPROCESSING_ELIGIBLE_RECORDING_MANIFEST.csv'
epoch_csv=TABLES/'10_LOCKED_AUTOMATIC_QA_EPOCH_MANIFEST.csv'
run_record={
 'experiment':'P1-E07.1 full-cohort CAUEEG reference preprocessing and automated epoch-QA lock',
 'timestamp_utc':pd.Timestamp.utcnow().isoformat(),'status':status,
 'source_p1_e06_run_record':str(E06_RUN_RECORD),'source_p1_e06_decision_sha256':sha256_file(E06_DECISION),
 'selected_path':'reference','input_recordings':input_count,'input_intervals':len(intervals),
 'preprocessing_eligible_recordings':eligible_count,'cohort_retention_fraction':retention,
 'processing_failure_fraction':failure_fraction,'selected_epochs':len(selected_epoch_manifest),
 'class_counts':{str(k):int(v) for k,v in eligible_manifest.experimental_label.value_counts().items()},
 'split_counts':{str(k):int(v) for k,v in eligible_manifest.official_split.value_counts().items()},
 'locked_recording_manifest_csv':str(recording_csv),'locked_recording_manifest_sha256':sha256_file(recording_csv),
 'locked_epoch_manifest_csv':str(epoch_csv),'locked_epoch_manifest_sha256':sha256_file(epoch_csv),
 'epoch_array_directory':str(ARRAYS),'epoch_array_files':eligible_count,
 'preprocessing':{'input_reference':'released AVG; no second CAR','bandpass_Hz':[1.0,40.0],
    'filter_order':FILTER_ORDER,'zero_phase':True,'target_sfreq_Hz':FS_TARGET,
    'edge_trim_seconds':EDGE_TRIM_SECONDS,'epoch_seconds':EPOCH_SECONDS,'overlap_fraction':0.0},
 'automatic_qa':{'ptp_limit_uV':PTP_LIMIT_UV,'rms_limit_uV':RMS_LIMIT_UV,
    'extreme_threshold_uV':EXTREME_UV,'maximum_extreme_fraction':MAX_EXTREME_FRACTION,
    'robust_score_limit':ROBUST_SCORE_LIMIT,'minimum_epochs':MIN_EPOCHS,'target_epochs':TARGET_EPOCHS,
    'manual_specialist_review_required':False},
 'identity_interpretation':{'analysis_unit':'recording','participant_identity_verified':False,
    'official_no_overlap_split_preserved':True},
 'python':sys.version,'platform':platform.platform(),'mne':mne.__version__}
(OUTPUT_ROOT/'run_record.json').write_text(json.dumps(run_record,indent=2),encoding='utf-8')
print(json.dumps(run_record,indent=2))
if not checks['pass'].all(): raise AssertionError(status)

## Next Steps

After a complete P1-E07 PASS:

1. Use only `09_LOCKED_PREPROCESSING_ELIGIBLE_RECORDING_MANIFEST.csv`, `10_LOCKED_AUTOMATIC_QA_EPOCH_MANIFEST.csv`, the authenticated epoch arrays and `run_record.json` downstream.
2. The next notebook should be **P1-E08. leakage-free non-spiking representation and benchmark construction**.
3. Fit every trainable transformation and normalization statistic using the training split only.
4. Preserve recording-level grouping until verified participant identifiers become available.
5. Do not use validation or test performance to revise preprocessing or automatic-QA thresholds.
6. Independent EEG-specialist review may be recommended as future validation, but it is not required for this automatic lock.